# 09 — DenseNet121 Preprocessing Screen @224

Mục tiêu: giữ nguyên DenseNet121 @224 và chỉ thay preprocessing để kiểm tra xem có cải thiện P trên fold 0 hay không.

Các biến thể:
1. baseline_raw — pad + resize như baseline hiện tại
2. clahe — CLAHE trên grayscale trước khi đưa thành 3-channel RGB
3. percentile_stretch — clip intensity theo percentile 1–99% rồi rescale về 0–255

Giữ cố định:
- locked fold 0 split
- ImageNet pretrained DenseNet121
- 224×224
- same augmentation
- weighted cross-entropy
- AdamW + cosine LR
- P@0.50 là tiêu chí screen chính
- holdout 770 ảnh không dùng

Baseline fold-0:
- P@0.50 = 0.9851505115
- tuned P = 0.9872839167

Memory-safe update: physical batch 8 + gradient accumulation 2 (effective batch 16), plus CUDA allocator cleanup.


In [ ]:
print('>>> CELL 1 START: imports/config', flush=True)
import os
os.environ['TORCH_HOME']=r'D:\AI_Cache\torch'

from pathlib import Path
import json, time, random, gc
import numpy as np
import pandas as pd
from PIL import Image, ImageOps

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from sklearn.metrics import confusion_matrix, f1_score, roc_auc_score

try:
    import cv2
except ImportError as e:
    raise RuntimeError('Missing OpenCV. Run in Terminal: python -m pip install opencv-python') from e

SEED=2026
ROOT=Path(r'D:\medicine')
MANIFEST=ROOT/'split_outputs'/'split_manifest.csv'
OUT=ROOT/'screen_outputs'/'densenet_preprocessing'
OUT.mkdir(parents=True,exist_ok=True)

FOLD=0
IMAGE_SIZE=224
BATCH_SIZE=8
GRAD_ACCUM_STEPS=2
EPOCHS=8
PATIENCE=3
LR=1e-4
WD=1e-4
PROGRESS_EVERY=50
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
USE_AMP=DEVICE.type=='cuda'
torch.backends.cudnn.benchmark=USE_AMP

REFERENCE_P_AT_05=0.9851505114783624
REFERENCE_TUNED_P=0.9872839166987125

CANDIDATES=['baseline_raw','clahe','percentile_stretch']

print('Device:',DEVICE,flush=True)
if USE_AMP: print('GPU:',torch.cuda.get_device_name(0),flush=True)
print('Candidates:',CANDIDATES,flush=True)
print('Batch:',BATCH_SIZE,'| accum:',GRAD_ACCUM_STEPS,'| effective:',BATCH_SIZE*GRAD_ACCUM_STEPS,flush=True)


In [ ]:
print('>>> CELL 2 START: load locked fold 0', flush=True)
m=pd.read_csv(MANIFEST)
dev=m[m.split=='dev'].copy().reset_index(drop=True)
holdout=m[m.split=='holdout'].copy().reset_index(drop=True)
train_df=dev[dev.cv_fold!=FOLD].copy().reset_index(drop=True)
val_df=dev[dev.cv_fold==FOLD].copy().reset_index(drop=True)

assert len(train_df)==3489 and len(val_df)==873 and len(holdout)==770
assert set(train_df.sha256).isdisjoint(set(val_df.sha256))
assert set(dev.sha256).isdisjoint(set(holdout.sha256))
print('Train:',len(train_df),'Val:',len(val_df),'SEALED holdout:',len(holdout),flush=True)


In [ ]:
print('>>> CELL 3 START: helpers/cache builders', flush=True)

class PadToSquare:
    def __call__(self,img):
        w,h=img.size; s=max(w,h)
        l=(s-w)//2; r=s-w-l; t=(s-h)//2; b=s-h-t
        return ImageOps.expand(img,border=(l,t,r,b),fill=0)

pad=PadToSquare()
MEAN=[0.485,0.456,0.406]
STD=[0.229,0.224,0.225]
L2I={'Normal':0,'Pneumonia':1}

def preprocess_pil(im,mode):
    im=im.convert('L')
    if mode=='clahe':
        a=np.asarray(im,dtype=np.uint8)
        clahe=cv2.createCLAHE(clipLimit=2.0,tileGridSize=(8,8))
        a=clahe.apply(a)
        im=Image.fromarray(a,mode='L')
    elif mode=='percentile_stretch':
        a=np.asarray(im,dtype=np.float32)
        lo,hi=np.percentile(a,[1,99])
        if hi>lo:
            a=np.clip((a-lo)/(hi-lo),0,1)*255.0
        a=np.asarray(a,dtype=np.uint8)
        im=Image.fromarray(a,mode='L')
    elif mode!='baseline_raw':
        raise ValueError(mode)
    return im.convert('RGB')

def build_cache(frame,mode):
    arr=[]
    print(f'Caching {mode}: {len(frame)} images...',flush=True)
    t0=time.time()
    for i,row in frame.iterrows():
        with Image.open(row.path) as im:
            im=preprocess_pil(im,mode)
            im=pad(im)
            im=im.resize((IMAGE_SIZE,IMAGE_SIZE),Image.Resampling.BILINEAR)
            arr.append(np.asarray(im,dtype=np.uint8))
        if (i+1)%500==0 or (i+1)==len(frame):
            print(f'  cached {i+1}/{len(frame)}',flush=True)
    out=np.stack(arr,axis=0)
    print(f'Cache {mode}: {out.nbytes/1024**3:.2f} GiB | {time.time()-t0:.0f}s',flush=True)
    return out

train_tf=transforms.Compose([
    transforms.RandomAffine(degrees=5,translate=(0.02,0.02),scale=(0.95,1.05),fill=0),
    transforms.ColorJitter(brightness=0.08,contrast=0.08),
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),
])
val_tf=transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN,STD),
])

class CachedDataset(Dataset):
    def __init__(self,frame,cache,tf):
        self.df=frame.reset_index(drop=True); self.cache=cache; self.tf=tf
    def __len__(self): return len(self.df)
    def __getitem__(self,i):
        row=self.df.iloc[i]
        return self.tf(Image.fromarray(self.cache[i])),L2I[row.label],row.filename

def metric(y,prob,t=0.5,auc_value=None):
    y=np.asarray(y,int); prob=np.asarray(prob,float); pred=(prob>=t).astype(int)
    tn,fp,fn,tp=confusion_matrix(y,pred,labels=[0,1]).ravel()
    acc=(tp+tn)/len(y); se=tp/max(tp+fn,1); sp=tn/max(tn+fp,1)
    f1=f1_score(y,pred,pos_label=1,zero_division=0)
    P=.30*acc+.25*se+.20*sp+.25*f1
    auc=roc_auc_score(y,prob) if auc_value is None else float(auc_value)
    return {'P':float(P),'Accuracy':float(acc),'Sensitivity':float(se),'Specificity':float(sp),
            'F1':float(f1),'AUROC':float(auc),'TN':int(tn),'FP':int(fp),'FN':int(fn),'TP':int(tp),
            'threshold':float(t)}

@torch.no_grad()
def predict(model,loader):
    model.eval(); ys=[]; ps=[]; names=[]
    for x,y,n in loader:
        x=x.to(DEVICE,non_blocking=True)
        with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
            logits=model(x)
        p=torch.softmax(logits.float(),1)[:,1]
        ys.extend(y.numpy()); ps.extend(p.cpu().numpy()); names.extend(list(n))
    return np.asarray(ys),np.asarray(ps),names

def make_model():
    m=models.densenet121(weights=models.DenseNet121_Weights.DEFAULT)
    m.classifier=nn.Linear(m.classifier.in_features,2)
    return m.to(DEVICE)

def threshold_scan(y,p):
    auc=float(roc_auc_score(y,p))
    ts=np.round(np.arange(0.10,0.9001,0.005),3)
    df=pd.DataFrame([metric(y,p,t,auc_value=auc) for t in ts])
    best=df.sort_values(['P','threshold'],ascending=[False,True]).iloc[0]
    return df,best


## Train preprocessing candidates on fold 0


In [ ]:
print('>>> CELL 5 START: preprocessing screen', flush=True)
results=[]

for mode in CANDIDATES:
    print('\n'+'='*80,flush=True)
    print('PREPROCESSING:',mode,flush=True)
    print('='*80,flush=True)

    random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
    if USE_AMP: torch.cuda.manual_seed_all(SEED)

    train_cache=build_cache(train_df,mode)
    val_cache=build_cache(val_df,mode)

    train_loader=DataLoader(CachedDataset(train_df,train_cache,train_tf),
                            batch_size=BATCH_SIZE,shuffle=True,num_workers=0,pin_memory=False)
    val_loader=DataLoader(CachedDataset(val_df,val_cache,val_tf),
                          batch_size=BATCH_SIZE,shuffle=False,num_workers=0,pin_memory=False)

    gc.collect()
    if USE_AMP: torch.cuda.empty_cache()

    model=make_model()
    counts=train_df.label.value_counts()
    n0=int(counts['Normal']); n1=int(counts['Pneumonia']); total=n0+n1
    cw=torch.tensor([total/(2*n0),total/(2*n1)],dtype=torch.float32,device=DEVICE)
    criterion=nn.CrossEntropyLoss(weight=cw)
    optimizer=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=WD)
    scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS)
    scaler=torch.amp.GradScaler('cuda',enabled=USE_AMP)

    best=-1.0; best_epoch=0; best_state=None; stale=0; hist=[]

    for epoch in range(1,EPOCHS+1):
        t0=time.time(); model.train(); loss_sum=0.0; seen=0
        for bi,(x,y,_) in enumerate(train_loader,1):
            x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            with torch.amp.autocast(device_type='cuda',enabled=USE_AMP):
                logits=model(x); loss=criterion(logits,y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
            loss_sum+=float(loss.item())*len(x); seen+=len(x)
            if bi%PROGRESS_EVERY==0 or bi==len(train_loader):
                print(f'  {mode} epoch {epoch:02d} batch {bi}/{len(train_loader)} | loss={loss_sum/seen:.4f}',flush=True)

        scheduler.step()
        yv,pv,_=predict(model,val_loader)
        mm=metric(yv,pv,0.5)
        sec=time.time()-t0
        hist.append({'mode':mode,'epoch':epoch,'train_loss':loss_sum/seen,'seconds':sec,**mm})
        print(f"{mode} Epoch {epoch:02d} | P={mm['P']:.5f} | A={mm['Accuracy']:.5f} | "
              f"Se={mm['Sensitivity']:.5f} | Sp={mm['Specificity']:.5f} | F1={mm['F1']:.5f} | "
              f"AUROC={mm['AUROC']:.6f} | {sec:.0f}s",flush=True)

        if mm['P']>best+1e-6:
            best=mm['P']; best_epoch=epoch; stale=0
            best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        else:
            stale+=1
            if stale>=PATIENCE:
                print(mode,'early stopping at epoch',epoch,flush=True)
                break

    model.load_state_dict(best_state)
    yv,pv,names=predict(model,val_loader)
    fixed=metric(yv,pv,0.5)
    scan,brow=threshold_scan(yv,pv)
    bt=float(brow.threshold); tuned=metric(yv,pv,bt)

    pd.DataFrame(hist).to_csv(OUT/f'{mode}_history.csv',index=False)
    pd.DataFrame({'filename':names,'y_true':yv,'p_pneumonia':pv}).to_csv(OUT/f'{mode}_predictions.csv',index=False)
    scan.to_csv(OUT/f'{mode}_threshold_scan.csv',index=False)
    torch.save({'architecture':'densenet121','preprocessing':mode,'image_size':224,'fold':0,
                'best_epoch':best_epoch,'state_dict':best_state},OUT/f'{mode}_best.pt')

    summary={'mode':mode,'best_epoch':best_epoch,'metrics_at_0_5':fixed,
             'best_threshold':bt,'tuned_metrics':tuned,'holdout_used':False}
    with open(OUT/f'{mode}_summary.json','w',encoding='utf-8') as f:
        json.dump(summary,f,indent=2)

    results.append({
        'mode':mode,'best_epoch':best_epoch,'P_at_0_5':fixed['P'],'AUROC':fixed['AUROC'],
        'Sensitivity':fixed['Sensitivity'],'Specificity':fixed['Specificity'],'F1':fixed['F1'],
        'best_threshold':bt,'P_tuned_fold0':tuned['P'],
        'delta_vs_reference_P_at_0_5':fixed['P']-REFERENCE_P_AT_05,
        'delta_vs_reference_tuned':tuned['P']-REFERENCE_TUNED_P
    })

    del model,optimizer,scheduler,scaler,best_state,train_loader,val_loader,train_cache,val_cache
    gc.collect()
    if USE_AMP: torch.cuda.empty_cache()

print('>>> PREPROCESSING SCREEN FINISHED',flush=True)


In [ ]:
print('>>> CELL 6 START: comparison', flush=True)
table=pd.DataFrame(results).sort_values('P_at_0_5',ascending=False)
table.to_csv(OUT/'preprocessing_screen_summary.csv',index=False)
display(table)

winner=table.iloc[0]
print('Best candidate:',winner.to_dict(),flush=True)

# Because baseline_raw here is a rerun, compare both against the historical 224 reference.
improved=table[table.P_at_0_5>REFERENCE_P_AT_05].sort_values('P_at_0_5',ascending=False)
if len(improved):
    print('PROMOTE FOR FURTHER VALIDATION:',improved.iloc[0]['mode'],flush=True)
else:
    print('No preprocessing candidate beat the historical 224 P@0.50 baseline.',flush=True)


## Decision rule

- P@0.50 là tiêu chí screen chính.
- Threshold-tuned P chỉ là tín hiệu phụ trên fold 0.
- Nếu CLAHE hoặc percentile stretch thắng rõ, mới đưa candidate sang 5-fold.
- Nếu không, giữ raw baseline và chuyển sang augmentation/loss/LR ablation.
